# Exercise 1: Tokens, sampling, and a tool by hand

**Lecture segment:** Lecture 1, Parts 2–3 (tokens, the next-token distribution, sampling); Lecture 2, Part 4.1 (a tool call by hand).
**Goal:** explain why a model can't count letters, how sampling turns a distribution into confident text, and how a few sentences in a system prompt give a model a tool.
**Models:** OLMo 2 1B (base), runs locally. Qwen3 30B on the SAIA API for Tasks 6–8 (needs `SAIA_API_KEY`).
**Time:** about 45 minutes.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")  # the shared helpers live in ../llm_helpers.py
sys.path.insert(0, "../demos")  # the tool loop from the lecture lives in ../demos/tool_loop.py
from collections import Counter
from llm_helpers import *
from tool_loop import by_hand
import matplotlib.pyplot as plt

base = load(BASE)  # OLMo 2 1B, only pretrained

## Task 1: What the model reads

A model never sees letters. The tokenizer cuts the text into tokens and the model gets their ID numbers.
Compare two tokenizers: OLMo's (100,278 tokens) and Qwen's, the API model (151,669 tokens).
Run the cell, then add texts of your own: your name, a German word, a number, a line of code.

In [ ]:
texts = ["1234567 * 7654321", "Donaudampfschifffahrtsgesellschaftskapitän", "strawberry", " strawberry", "    return x"]  # TODO: add texts of your own

for text in texts:
    print("OLMo:", end=" ")
    show_tokens(base.tok, text)
    print("Qwen:", end=" ")
    show_tokens(api_tokenizer(), text)

**Observe:** Which tokenizer cuts numbers into single digits? Is `strawberry` the same token as ` strawberry`?
The model gets `Donaudampfschifffahrtsgesellschaftskapitän` as a handful of ID numbers. Why is "How many times does the letter s occur in this word?" a hard question for it?

*Your answer:*

## Task 2: One distribution, many confident answers

The model outputs a probability for every possible next token. The text you see is one draw from it.
First look at the distribution, then draw 20 times and count what comes out. (The prompt ends with a space on purpose: then the next token is the start of the year.)

In [ ]:
prompt = "The Technical University of Munich was founded in "  # TODO: try "The capital of Australia is"
show_top(base, prompt, k=10)

In [ ]:
draws = Counter()
for seed in range(20):
    text = complete(base, prompt, max_new_tokens=3, temperature=1.0, seed=seed)
    draws[text.split()[0].strip(".,;") if text.split() else text] += 1  # the first word of the continuation
draws.most_common()

**Observe:** How often did the correct year (1868) come out? Compare the counts with the probabilities above.
Each draw, written as a sentence, would state its year with the same confidence. What does that tell you about a single answer from a chat model?

*Your answer:*

## Task 3: Temperature reshapes the distribution

Temperature divides the model's raw scores (logits) before they are turned into probabilities.
Run the cell, then change the prompt and the temperatures and run it again.

In [ ]:
prompt = "My favourite food is"  # TODO: try a prompt of your own
temperatures = [0.3, 1.0, 2.0]   # TODO: try other values

logits = base(encode(base, prompt)).logits[0, -1].float().detach()
fig, axes = plt.subplots(1, len(temperatures), figsize=(14, 3.5), sharey=True)
for ax, T in zip(axes, temperatures):
    p, ids = (logits / T).softmax(-1).topk(12)
    ax.bar([base.tok.decode(i) for i in ids.tolist()], p.cpu())
    ax.set_yscale("log")
    ax.set_ylim(1e-4, 1)
    ax.set_title(f"temperature = {T}: top 12 hold {p.sum():.0%}")
    ax.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("probability (log scale)")
plt.tight_layout()

**Observe:** The order of the tokens never changes. What does change? Where does the probability go that is not in the top 12?

*Your answer:*

## Task 4: Temperature in generated text

The same prompt at several temperatures, with two random seeds each. `temperature=0` means greedy: always the most likely token.

In [ ]:
prompt = "Once upon a time, in a small village in the Alps,"
temperatures = [0.0, 0.7, 1.5]  # TODO: find the temperature where the text starts to fall apart

for T in temperatures:
    print(f"--- temperature {T}")
    for seed in [1, 2]:
        print(repr(complete(base, prompt, max_new_tokens=30, temperature=T, seed=seed)))

**Observe:** Why are both seeds identical at temperature 0, and different at higher temperatures?

*Your answer:*

## Task 5: top-p cuts off the tail

top-p (nucleus sampling) keeps only the most likely tokens whose probabilities add up to p, and samples among those.
Here the temperature is high on purpose.

In [ ]:
top_ps = [1.0, 0.9, 0.5]  # TODO: try other values between 0 and 1

for p in top_ps:
    print(f"top_p={p}:", repr(complete(base, prompt, max_new_tokens=30, temperature=1.5, top_p=p, seed=1)))

**Observe:** Why does a lower top-p rescue the text at high temperature? Chat services often use a temperature around 0.7 with top-p around 0.9: what does each setting do?

*Your answer:*

## Task 6: A model that can't count

The API model is a 30B model, trained for chat and for tools. Ask it to count letters.
The correct answer is 5.

In [ ]:
question = "How many times does the letter s occur in the word Donaudampfschifffahrtsgesellschaftskapitän?"
print(ask(question + " Answer with a number only."))  # TODO: then remove the instruction and run again

**Observe:** Is the number right? Without "Answer with a number only.", what does the model do instead, and why could that help (Task 1)? Does it finish?

*Your answer:*

## Task 7: Give the model a letter counter

Counting letters is one line of Python. Here is the tool:

In [ ]:
def count_letter(word, letter):
    return word.count(letter)

count_letter("strawberry", "r")

The harness is `by_hand()` from the lecture (`demos/tool_loop.py`). It writes the whole document in Qwen's chat format: your system prompt, the question, and `<|im_start|>assistant`. Then it lets the model continue. When the model writes

```
<tool_call>{"name": "count_letter", "arguments": {"word": "...", "letter": "..."}}</tool_call>
```

the harness runs `count_letter(word=..., letter=...)`, writes the result into the document as `<tool_response>…</tool_response>`, and lets the model continue again.

Your job: write the system prompt that makes the model write such a call. The calculator prompt from the lecture (4.1) shows how:

> You can use a calculator. To use it, write `<tool_call>{"name": "calc", "arguments": {"expression": "..."}}</tool_call>` and stop. The result will be given to you.

The names have to match exactly: `count_letter`, `word`, `letter`. If they don't, you get a Python error that says which name is wrong. The cell runs as it is, but the placeholder prompt describes no tool.

In [ ]:
system = "You are a helpful assistant."  # TODO: describe the letter counter
tools = {"count_letter": count_letter}

doc = by_hand(question, system, tools)
print(doc)

**Observe:** The printed text is the whole document. Which parts did the model write, and which did the harness write? Where is the number 5 computed?

*Your answer:*

## Task 8: Does the model know when to use it?

Run your prompt on more questions. The last one doesn't need the tool.

In [ ]:
questions = [
    "How many f's are in Donaudampfschifffahrtsgesellschaftskapitän?",
    "How many e's are in Rindfleischetikettierungsüberwachungsaufgabenübertragungsgesetz?",
    "How many r's are in the word strawberry?",
    "What is the capital of Australia?",
]  # TODO: add questions of your own

for q in questions:
    doc = by_hand(q, system, tools)
    print(q)
    print("   ", doc.split("<|im_start|>assistant\n", 1)[1].replace("\n", "\n    "))  # only what came after the question
    print()

**Observe:** Did the model call the tool only when it helps? Compare each tool result with the number in the final answer: does the model always copy it correctly?

*Your answer:*

## Bonus: an optional argument

Ask about upper and lower case.

In [ ]:
print(by_hand("How many a's, upper or lower case, are in the word Aachen?", system, tools))

What did the tool return, and what did the model answer? Now give the tool an optional argument `ignore_case`.
Complete the function (hint: `word.lower()` makes a word lower case), then describe the new argument in your system prompt, so that the model knows when to set `"ignore_case": true`. Run the question above again.

In [ ]:
def count_letter(word, letter, ignore_case=False):
    # TODO: if ignore_case is True, make word and letter lower case
    return word.count(letter)

tools = {"count_letter": count_letter}
print(count_letter("Aachen", "a"), count_letter("Aachen", "a", ignore_case=True))  # should print: 1 2

**Observe:** Does the model set `ignore_case` when the question asks for it, and leave it out otherwise?

*Your answer:*

## Wrap-up

In one or two sentences: what did the model do in Task 7, and what did the program around it do?

*Your answer:*